### 1. Basic Tasks

### 1.
Warehouse 
- A Data Warehouse is a centralized storage used to store clean, structured and organised data following a predefined schema for analytics nd BI purposes

Lake
- A Data Lake is a centralised storage the can store large volume of raw data. It supports structured, semi structured and unstructured data. Suitable for big data, data science and machine learning purposes.

Lakehouse 
- A Data Lakehouse is a centralized storage that combines both features data warehouse and lake. It supports structured, semi structured and unstructured data. It combines flexiblity of data lake and analytics like data warehouse. 

In [0]:
-- 2. 
create table if not exists dev.demo.products (product_id int, name string, category string, price double);

insert into dev.demo.products values (1, 'Laptop', 'Electronics', 799.99),
(2, 'Mouse', 'Electronics', 24.99),
(3, 'Keyboard', 'Electronics', 49.99),
(4, 'Monitor', 'Electronics', 199.99),
(5, 'Desk Chair', 'Furniture', 149.99),
(6, 'Desk', 'Furniture', 249.99),
(7, 'Notebook', 'Stationery', 5.99),
(8, 'Pen Set', 'Stationery', 9.99),
(9, 'Backpack', 'Accessories', 39.99),
(10, 'Water Bottle', 'Accessories', 14.99);

In [0]:
-- 3
update dev.demo.products set price = 15.99 where product_id = 10

In [0]:
insert into dev.demo.products values (11, "Laptop", "Electronics", 499.99)

In [0]:
select * from dev.demo.products

In [0]:
desc history dev.demo.products

In [0]:
-- 4
select * from dev.demo.products version as of 1

Latest version has 11 records with price update for product id 10 while version 1 had only 10 records and no price update

### 2. Intermediate Tasks

In [0]:
%python
# 5
# Deliberatly insserting raw with extra column throws error "Delta meatadata mismatched"
schema = "product_id int, name string, category string, price double, stock int"
data = [(12,"Lunch box", "Accessories", 14.99, 10)]
df = spark.createDataFrame(data, schema)
df.write.mode("overwrite").saveAsTable("dev.demo.products")

In [0]:
%python
# Implemented merge schema before inserting row with extra column, that evolves schema to include extra column
schema = "product_id int, name string, category string, price double, stock int"
data = [(12,"Lunch box", "Accessories", 14.99, 10)]
df_merge = spark.createDataFrame(data, schema)
df_merge.write.mode("append").option("mergeSchema",True).saveAsTable("dev.demo.products")

In [0]:
select * from dev.demo.products 

In [0]:
desc history dev.demo.products

In [0]:
-- 6
select * from dev.demo.products version as of 3

In [0]:
restore table dev.demo.products to version as of 3

When multiple data pipelines writes or updates same table at same time, there could be a risk that the 2 write operation could conflict and leave table in inconsistent or incomplete state.  
ACID transaction refers to : 
- Atomicity: Ensures that a transaction is either fully executed or not at all. If a part of transaction fails, it rolls back to the previous stable state instead of leaving database at an inconsistent state with partial updates 
- Consistency: The data must follow the schema, constraints and defined rules, if the data do not follow the rule is termed as bad data and automatically rejected to maintain the consistency of the schema of the database 
- Isolation: Ensures every transaction runs independently so that concurrent operations do not conflict each other  
- Durability: Once a transaction is successfully committed, the changes are permanently stored and are not lost if the system fails. 
 
If 2 pipelines updates same table simultaneously, ACID transactions helps ensure that users do not see partially written data or inconsistent. This makes data reliable and consistent even when multiple pipelines are running concurrently 

### 3. Advanced Tasks

8

Cyntexa could replace the existing nightly batch warehouse using a lakehouse pipeline  with a medallions architecture implementation where they ingest data incrementally or continuously in bronze layer, clean and transform it in silver layer and produce business ready KPIs in gold layer. This would reduce dependency o a single large nightly batch load.

The Lakehouse use ACID transactions to make writes and updates reliable and consistent. If multiple pipeline writes or updates same delta table it prevents partial or conflicting updates from leaving table in an inconsistent state.

Time Travel reduce operational risk by allowing access to previous versions. If an incorrect update or accidental data  change occurs, we can inspect the versions and restore the data to stable state 

In [0]:
-- 9
desc history dev.demo.products

In [0]:
select * from dev.demo.products version as of 9


A lakehouse provides several advantages to data analysts compared with a traditional data warehouse. It combines the flexibility of a data lake with the reliability and analytical capabilities of a data warehouse.

1. Access to More Types of Data

A traditional warehouse mainly works with structured and cleaned data. A lakehouse can support **structured, semi-structured, and unstructured data** in the same platform. This allows analysts to work with sources such as CSV files, JSON data, application logs, and other raw data without requiring everything to be converted into a traditional warehouse format first.

2. Access to More Up-to-Date Data

Traditional warehouses often depend on scheduled batch jobs, such as nightly loads, before new data becomes available to analysts. A lakehouse can support **incremental and streaming data pipelines**, allowing data to be updated more frequently. This means analysts can work with more recent data and build dashboards and reports that reflect business changes sooner.

3. Better Data Reliability and Recovery

Lakehouses commonly use **ACID transactions and features such as Time Travel**. ACID transactions help ensure that analysts do not see partially completed or inconsistent updates when multiple pipelines modify the same data. Time Travel allows previous versions of data to be accessed, making it easier to investigate incorrect updates or recover from accidental changes.

Tradeoff to Watch For: Complexity

Although a lakehouse provides greater flexibility, it can introduce additional complexity. Analysts may have access to raw, cleaned, and aggregated versions of the same data, and they need to understand which tables are appropriate for analysis. Without proper data governance, documentation, and access controls, this flexibility can lead to confusion, inconsistent analysis, or incorrect results.

